In [1]:
print("ok")

ok


In [2]:
  
import os                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
GOOGLE_API_KEY=os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
LANGCHAIN_API_KEY=os.getenv("LANGCHAIN_API_KEY")
LANGCHAIN_PROJECT=os.getenv("LANGCHAIN_PROJECT")

In [3]:
LANGCHAIN_PROJECT

'langgraph-agent'

In [4]:
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY
os.environ["GROQ_API_KEY"]= GROQ_API_KEY
os.environ["LANGCHAIN_API_KEY"] = LANGCHAIN_API_KEY
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"]=LANGCHAIN_PROJECT

In [11]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

In [5]:
'''from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")'''
from langchain_groq import ChatGroq
import os
llm=ChatGroq(model_name="llama-3.1-8b-instant")

f:\Conda\envs\langgraph_tut_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Simple AI Assistant

In [6]:
while True:
    question=input("type your question. if you want to quit the chat write quit")
    if question !="quit":
        print(llm.invoke(question).content)
    else:
        print("goodbye take care yourself")
        break

How can I assist you today?
Nice to meet you, Tushar. How can I assist you today? Do you have a question, or would you like to have a conversation?
I don't have any information about your name. I'm a conversational AI, and our conversation just started. I don't have any prior knowledge or context about you. If you'd like to share your name, I'd be happy to chat with you about it.
goodbye take care yourself


## Adding Memory to the Assistant

In [20]:

from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import AIMessage

In [21]:
store={}

In [22]:
store={}
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [23]:
config = {"configurable": {"session_id": "firstchat"}}

In [24]:
model_with_memory=RunnableWithMessageHistory(llm,get_session_history)

In [25]:
model_with_memory.invoke(("Hi! I'm Tushar Choudhary"),config=config).content

'Nice to meet you, Tushar Choudhary. Is there something I can help you with today?'

In [26]:

model_with_memory.invoke(("tell me what is my name?"),config=config).content

'Your name is Tushar Choudhary.'

In [27]:

store

{'firstchat': InMemoryChatMessageHistory(messages=[HumanMessage(content="Hi! I'm Tushar Choudhary", additional_kwargs={}, response_metadata={}), AIMessage(content='Nice to meet you, Tushar Choudhary. Is there something I can help you with today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 46, 'total_tokens': 70, 'completion_time': 0.036887691, 'completion_tokens_details': None, 'prompt_time': 0.002865754, 'prompt_tokens_details': None, 'queue_time': 0.046365086, 'total_time': 0.039753445}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cd77e-93a9-71d2-9872-c55a5eecc7c5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 46, 'output_tokens': 24, 'total_tokens': 70}), HumanMessage(content='tell me what is my name?', additional_kwargs={}, response_metadata={}), AIMessag

## RAG with LCEL

In [30]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [31]:
### Reading the txt files from source directory

loader = DirectoryLoader('../data', glob="./*.txt", loader_cls=TextLoader)
docs = loader.load()

In [34]:
type(docs)

list

In [35]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10,
    length_function=len
)
new_docs = text_splitter.split_documents(documents=docs)

In [36]:
doc_strings = [doc.page_content for doc in new_docs]

In [39]:
###  BGE Embddings

from langchain_community.embeddings import HuggingFaceBgeEmbeddings

model_name = "BAAI/bge-base-en-v1.5"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': True} # set True to compute cosine similarity
embeddings = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)


### Creating Retriever using Vector DB

db = Chroma.from_documents(new_docs, embeddings)
retriever = db.as_retriever(search_kwargs={"k": 4})

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_7792\2313199524.py:8: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceBgeEmbeddings(
f:\Conda\envs\langgraph_tut_venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--BAAI--bge-base-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see

In [40]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = PromptTemplate.from_template(template)

In [41]:
retrieval_chain = (
    RunnableParallel({"context": retriever, "question": RunnablePassthrough()})
    | prompt
    | llm
    | StrOutputParser()
    )

In [42]:
question ="what is llama3? can you highlight 3 important points?"
print(retrieval_chain.invoke(question))

Based on the provided context, Llama 3 appears to be a large language model developed by Meta. Here are 3 important points related to Llama 3:

1. **Release Date**: The release date of Llama 3 is unclear from the given context, but it mentions "released in April 2024" in one of the documents.

2. **Parameter Version**: There's a mention of the "8B parameter version of Llama 3." This suggests that Llama 3 comes in different parameter sizes, with 8B being one of the versions.

3. **Usage in Services**: Llama 3 is used in some Meta services, including a website and another service, although the name of the other service is not specified.


## Let's Start with Tools and Agents

In [43]:

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [44]:
api_wrapper=WikipediaAPIWrapper()

In [45]:
tool=WikipediaQueryRun(api_wrapper=api_wrapper)

In [46]:
tool.name

'wikipedia'

In [47]:
tool.description

'A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.'

In [48]:

tool.args

{'query': {'description': 'query to look up on wikipedia',
  'title': 'Query',
  'type': 'string'}}

In [49]:
print(tool.run({"query": "langchain"}))

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.

Page: Vector database
Summary: A vector database, vector store or vector search engine is a database that stores and retrieves embeddings of data in vector space. Vector databases typically implement approximate nearest neighbor algorithms so users can search for records semantically similar to a given input, unlike traditional databases which primarily look up records by exact match. Use-cases for vector databases include similarity search, semantic search, multi-modal search, recommendations engines, object detection, and retrieval-augmented generation (RAG).
Vector embeddings are mathematical representations of data in a high-dimensional spa

In [54]:
from langchain_community.tools import YouTubeSearchTool

In [55]:
tool2=YouTubeSearchTool()

In [56]:
tool2.name

'youtube_search'

In [57]:
tool2.description

'search for youtube videos associated with a person. the input to this tool should be a comma separated list, the first part contains a person name and the second a number that is the maximum number of video results to return aka num_results. the second part is optional'

In [58]:
tool2.run("sunny savita")

"['https://www.youtube.com/watch?v=ENzZuvahKwc&pp=ygUMc3Vubnkgc2F2aXRh', 'https://www.youtube.com/watch?v=OtdnFtHOu40&pp=ygUMc3Vubnkgc2F2aXRh']"

In [59]:
from langchain_community.tools.tavily_search import TavilySearchResults

In [60]:
tool3=TavilySearchResults()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_7792\140006202.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tool3=TavilySearchResults()


In [61]:
tool.invoke({"query": "What happened in the latest burning man floods"})

'Page: Burning Man\nSummary: Burning Man is a week-long large-scale desert event focused on "community, art, self-expression, and self-reliance" held annually in the Western United States. The event\'s name comes from its ceremony on the second last night of the event: the symbolic burning of a large wooden effigy, referred to as the Man, the Saturday evening before Labor Day. Since 1990, the event has been at Black Rock City in northwestern Nevada, a temporary city erected in the Black Rock Desert about 100 miles (160 km) north-northeast of Reno. According to Burning Man co-founder Larry Harvey in 2004, the event is guided by ten stated principles: radical inclusion, gifting, decommodification, radical self-reliance, radical self-expression, communal effort, civic responsibility, leaving no trace, participation, and immediacy. Burning Man belongs among the \'Big Five\' of the international event circuit—along with the Full Moon Party, Zurich’s Street Parade, Ibiza, and the Rio Carniva

### Adding langchain agent, Tool and initializing agent executor

In [7]:
import langchain
print(langchain.__version__)

1.2.10


In [ ]:
from langchain.agents import AgentType
from langchain.agents import load_tools
from langchain.agents import initialize_agent

In [ ]:
tool=load_tools(["wikipedia"],llm=llm)

In [ ]:
agent=initialize_agent(tool,llm,agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,verbose=True)

In [ ]:
agent.run("What is current GDP of India?")